# Train DataLoader 검증 노트북

`new_dataset.py` / `new_datamodule.py` train dataloader를 **셀 순서대로 실행**하여 end-to-end로 검증합니다.

| Step | 내용 |
|------|------|
| **Colab** | Drive 마운트 + `git clone` + pip install (**Colab에서만**) |
| 0 | 환경 설정 (`data/` → Drive 연결) |
| 1 | Zenodo에서 DCASE dev set 다운로드 → `data/dev_set/` |
| 2 | SpAudSyn clone → `src/modules/spatial_audio_synthesizer/` |
| 3 | Semantic Hearing `BinauralCuratedDataset` 다운로드 |
| 4 | Semantic Hearing 데이터를 dev_set에 추가 |
| 5 | 경로 검증 |
| 6 | `tests/datamodules` unittest |
| 7–8 | train / val dataloader 스모크 테스트 |

> **로컬 PC**: Colab 셀 skip → Step 0부터 (`data/`는 프로젝트 로컬 폴더).
>
> **Google Colab**: Colab 셀 필수. `data/`는 **`MyDrive/dcase2026_task4/data/`**에 저장되고 `PROJECT_ROOT/data` 심볼릭 링크로 연결됩니다.

## [Colab 전용] Google Drive 마운트 + repo clone

로컬 PC에서 실행 중이면 **이 셀을 skip**하세요.

- Drive: `/content/drive/MyDrive/dcase2026_task4/data/` (다운로드 데이터 영구 저장)
- Code: `/content/dcase2026_task4/` (`git clone`)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Mockdd/dcase2026_task4.git"
COLAB_REPO_DIR = Path("/content/dcase2026_task4")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/dcase2026_task4")
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive data dir:", DRIVE_DATA_DIR)

    if not (COLAB_REPO_DIR / "src" / "datamodules" / "new_dataset.py").exists():
        !git clone --depth 1 {REPO_URL} {COLAB_REPO_DIR}
    %cd {COLAB_REPO_DIR}

    !pip install -q lightning pyyaml soundfile librosa scipy python-sofa tqdm

    print("Colab setup OK")
    print("  code :", COLAB_REPO_DIR)
    print("  data :", DRIVE_DATA_DIR, "(persisted on Google Drive)")
else:
    print("Not Colab — skip Drive mount / clone.")

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import tarfile
import unittest
import urllib.request
import zipfile
from io import StringIO
from pathlib import Path

import torch
import yaml

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def _data_is_scaffold_only(data_path: Path) -> bool:
    """Step 0가 만든 빈 downloads/zenodo 폴더만 있으면 True."""
    if not data_path.exists():
        return True
    has_file = False
    for p in data_path.rglob("*"):
        if p.is_file() and p.stat().st_size > 0:
            has_file = True
            break
    return not has_file


if IN_COLAB:
    PROJECT_ROOT = Path("/content/dcase2026_task4")
    DATA_DIR = Path("/content/drive/MyDrive/dcase2026_task4/data")
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src" / "datamodules" / "new_dataset.py").exists():
        PROJECT_ROOT = Path.cwd().parent
    DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

LOCAL_DATA = PROJECT_ROOT / "data"
if IN_COLAB:
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    if LOCAL_DATA.is_symlink() and LOCAL_DATA.resolve() == DATA_DIR.resolve():
        print("[ok] symlink already set")
    elif _data_is_scaffold_only(LOCAL_DATA):
        if LOCAL_DATA.exists() or LOCAL_DATA.is_symlink():
            if LOCAL_DATA.is_symlink():
                LOCAL_DATA.unlink()
            else:
                shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.symlink_to(DATA_DIR, target_is_directory=True)
        print("[ok] removed empty scaffold, created symlink")
    else:
        print("[info] moving existing local data/ to Google Drive ...")
        shutil.copytree(LOCAL_DATA, DATA_DIR, dirs_exist_ok=True)
        shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.symlink_to(DATA_DIR, target_is_directory=True)
        print("[ok] data moved to Drive and symlink created")

    assert LOCAL_DATA.resolve() == DATA_DIR.resolve(), "symlink setup failed"
    print("symlink:", LOCAL_DATA, "->", DATA_DIR.resolve())

DEV_SET_DIR = DATA_DIR / "dev_set"
DOWNLOAD_DIR = DATA_DIR / "downloads"
ZENODO_DIR = DOWNLOAD_DIR / "zenodo"
SEMHEAR_TAR = DATA_DIR / "BinauralCuratedDataset.tar"
SEMHEAR_DIR = DATA_DIR / "BinauralCuratedDataset"
SPAUDSYN_DIR = PROJECT_ROOT / "src" / "modules" / "spatial_audio_synthesizer"

for d in (DATA_DIR, DOWNLOAD_DIR, ZENODO_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_DIR    :", DATA_DIR.resolve())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

In [ ]:
def describe_batch(batch, title="batch"):
    print(f"\n=== {title} ===")
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k:20s} shape={tuple(v.shape)} dtype={v.dtype}")
        elif isinstance(v, list):
            print(f"  {k:20s} list[len={len(v)}] first={v[0] if v else None}")
        else:
            print(f"  {k:20s} {type(v).__name__}")


def download_url(url: str, dest: Path, label: str = "") -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {label or dest.name} (already exists)")
        return

    print(f"[download] {label or dest.name}")
    print(f"  from {url}")

    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            done = min(block_num * block_size, total_size)
            print(f"\r  {done / total_size * 100:5.1f}% ({done // (1024**2)} MB)", end="", flush=True)

    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print("\n  done.")


def ensure_zip_tools() -> None:
    """Split zip merge/extract needs system zip + unzip (README: zip -s 0)."""
    if shutil.which("zip") and shutil.which("unzip"):
        return
    print("[install] apt install zip unzip")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zip", "unzip"], check=True)


def is_valid_zip(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with zipfile.ZipFile(path, "r") as zf:
            bad = zf.testzip()
            return bad is None
    except zipfile.BadZipFile:
        return False


def merge_split_zip(
    work_dir: Path,
    split_zip_name: str = "DCASE2026Task4Dataset.zip",
    merged_name: str = "DCASE2026Task4DatasetFull.zip",
) -> Path:
    """Merge Zenodo split zip via `zip -s 0` (NOT binary concat).

    Python zipfile cannot read multi-disk archives created by naive concatenation.
    See README: zip -s 0 DCASE2026Task4Dataset.zip --out DCASE2026Task4DatasetFull.zip
    """
    ensure_zip_tools()
    work_dir = Path(work_dir)
    split_zip = work_dir / split_zip_name
    merged_zip = work_dir / merged_name

    z_parts = sorted(work_dir.glob(split_zip_name.replace(".zip", ".z*")))
    if not split_zip.exists():
        raise FileNotFoundError(f"Missing last split part: {split_zip}")
    if not z_parts:
        raise FileNotFoundError(f"Missing .z01+ parts in {work_dir}")

    if merged_zip.exists():
        if is_valid_zip(merged_zip):
            print(f"[skip] valid merged zip: {merged_zip.name}")
            return merged_zip
        print(f"[warn] deleting invalid merged zip (re-merge with zip -s 0): {merged_zip.name}")
        merged_zip.unlink()

    print(f"[merge] zip -s 0 {split_zip_name} -> {merged_name}  ({len(z_parts)} .z* + .zip)")
    subprocess.run(
        ["zip", "-s", "0", split_zip_name, "--out", merged_name],
        cwd=work_dir,
        check=True,
    )
    if not is_valid_zip(merged_zip):
        raise zipfile.BadZipFile(
            f"Merge failed: {merged_zip} is not a valid zip. "
            "Check that all .z01-.z10 and .zip parts downloaded completely."
        )
    print("  merge done.")
    return merged_zip


def extract_zip(zip_path: Path, dest_dir: Path) -> None:
    marker = dest_dir / ".extracted"
    if marker.exists():
        print(f"[skip] already extracted: {dest_dir}")
        return

    ensure_zip_tools()
    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"[extract] {zip_path.name} -> {dest_dir}")
    subprocess.run(["unzip", "-q", str(zip_path), "-d", str(dest_dir)], check=True)
    marker.write_text("ok", encoding="utf-8")
    print("  extract done.")


def install_dev_set_from_extract(extract_root: Path) -> None:
    if (DEV_SET_DIR / "metadata" / "valid.json").exists():
        print(f"[skip] dev_set already at {DEV_SET_DIR}")
        return

    candidates = [
        extract_root / "DCASE2026Task4Dataset" / "data" / "dev_set",
        extract_root / "data" / "dev_set",
    ]
    candidates += [p for p in extract_root.rglob("dev_set") if (p / "metadata" / "valid.json").exists()]

    src = next((p for p in candidates if p.exists()), None)
    if src is None:
        raise FileNotFoundError(f"dev_set not found under {extract_root}")

    print(f"[install] {src} -> {DEV_SET_DIR}")
    DEV_SET_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DEV_SET_DIR.exists():
        shutil.rmtree(DEV_SET_DIR)
    shutil.copytree(src, DEV_SET_DIR)
    print("  dev_set installed.")

## Step 1. Zenodo DCASE dev set 다운로드

[Zenodo record 19328046](https://zenodo.org/records/19328046)의 split zip(`.z01`–`.z10` + `.zip`)을 받아 합친 뒤 `data/dev_set/`에 배치합니다.

README 절차: `wget -i dev_set_zenodo.txt` → `zip -s 0 ...` → `unzip`

In [ ]:
ZENODO_URLS = [
    line.strip()
    for line in (PROJECT_ROOT / "dev_set_zenodo.txt").read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().lower().endswith("lisence.pdf")
]

for url in ZENODO_URLS:
    fname = url.rsplit("/", 1)[-1]
    download_url(url, ZENODO_DIR / fname, label=fname)

# 이전 실행에서 binary concat으로 만든 깨진 merged zip / 실패한 extract 정리
merged_zip = ZENODO_DIR / "DCASE2026Task4DatasetFull.zip"
extract_root = ZENODO_DIR / "extracted"
if merged_zip.exists() and not is_valid_zip(merged_zip):
    print(f"[cleanup] remove bad merged zip: {merged_zip.name}")
    merged_zip.unlink()
if not (DEV_SET_DIR / "metadata" / "valid.json").exists():
    marker = extract_root / ".extracted"
    if marker.exists() and not any(extract_root.glob("**/valid.json")):
        print("[cleanup] remove stale extract marker")
        marker.unlink()

merged_zip = merge_split_zip(ZENODO_DIR)
extract_zip(merged_zip, extract_root)
install_dev_set_from_extract(extract_root)

assert (DEV_SET_DIR / "metadata" / "valid.json").exists(), "dev_set install failed"
print("\nStep 1 OK:", DEV_SET_DIR.resolve())

## Step 2. SpAudSyn 설치

train `mode: generate`에 필요합니다. [SpAudSyn](https://github.com/nttcslab/SpAudSyn) repo의 `src/`를 복사합니다.

In [ ]:
SPAUDSYN_MARKER = SPAUDSYN_DIR / "spatial_audio_synthesizer.py"

if SPAUDSYN_MARKER.exists():
    print(f"[skip] SpAudSyn already at {SPAUDSYN_DIR}")
else:
    clone_dir = DOWNLOAD_DIR / "SpAudSyn_repo"
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    print("[git clone] https://github.com/nttcslab/SpAudSyn.git")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/nttcslab/SpAudSyn.git", str(clone_dir)],
        check=True,
    )
    src_dir = clone_dir / "src"
    if not src_dir.exists():
        raise FileNotFoundError(f"SpAudSyn src/ not found in {clone_dir}")
    SPAUDSYN_DIR.parent.mkdir(parents=True, exist_ok=True)
    if SPAUDSYN_DIR.exists():
        shutil.rmtree(SPAUDSYN_DIR)
    shutil.copytree(src_dir, SPAUDSYN_DIR)
    print(f"[install] SpAudSyn -> {SPAUDSYN_DIR}")

assert SPAUDSYN_MARKER.exists(), "SpAudSyn install failed"
print("\nStep 2 OK")

## Step 3. Semantic Hearing 데이터 다운로드

[SemanticHearing README](https://github.com/vb000/SemanticHearing) Training 섹션:
`wget -P data https://semantichearing.cs.washington.edu/BinauralCuratedDataset.tar`

In [ ]:
SEMHEAR_URL = "https://semantichearing.cs.washington.edu/BinauralCuratedDataset.tar"

if SEMHEAR_DIR.exists() and any(SEMHEAR_DIR.iterdir()):
    print(f"[skip] already extracted: {SEMHEAR_DIR}")
elif SEMHEAR_TAR.exists():
    print(f"[extract] {SEMHEAR_TAR}")
    with tarfile.open(SEMHEAR_TAR, "r") as tar:
        tar.extractall(path=DATA_DIR)
else:
    download_url(SEMHEAR_URL, SEMHEAR_TAR, label="BinauralCuratedDataset.tar")
    print(f"[extract] {SEMHEAR_TAR}")
    with tarfile.open(SEMHEAR_TAR, "r") as tar:
        tar.extractall(path=DATA_DIR)

if not SEMHEAR_DIR.exists():
    alt = next((p for p in DATA_DIR.glob("Binaural*") if p.is_dir()), None)
    if alt:
        SEMHEAR_DIR = alt

assert SEMHEAR_DIR.exists(), "Semantic Hearing extract failed"
print("Contents:", [p.name for p in SEMHEAR_DIR.iterdir()][:12])
print("\nStep 3 OK:", SEMHEAR_DIR.resolve())

## Step 4. Semantic Hearing → dev_set 추가

`add_data.sh --semhear_path ...` 와 동일한 작업을 Python 스크립트로 실행합니다 (Windows 호환).

In [ ]:
SEMHEAR_ADD_MARKER = DEV_SET_DIR / ".semhear_added"
ADD_ARGS = {
    "target_sr": "32000",
    "amp_threshold": "0.02",
    "min_length": "0.1",
    "segment": "10",
    "shift": "0.1",
}


def run_py_script(script: str, extra_args: list[str]) -> None:
    cmd = [sys.executable, script] + extra_args
    print("[run]", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


if SEMHEAR_ADD_MARKER.exists():
    print("[skip] Semantic Hearing already added to dev_set")
else:
    bg_dir = SEMHEAR_DIR / "bg_scaper_fmt"
    fsd_dir = SEMHEAR_DIR / "FSD50K"
    assert bg_dir.exists(), f"missing {bg_dir}"
    assert fsd_dir.exists(), f"missing {fsd_dir}"

    common = [
        f"--target_sr={ADD_ARGS['target_sr']}",
        f"--amp_threshold={ADD_ARGS['amp_threshold']}",
        f"--min_length={ADD_ARGS['min_length']}",
        f"--segment={ADD_ARGS['segment']}",
        f"--shift={ADD_ARGS['shift']}",
    ]

    run_py_script(
        "add_interference.py",
        ["--input_dir", str(bg_dir), "--output_dir", str(DEV_SET_DIR / "interference"), *common],
    )
    run_py_script(
        "add_sound_event.py",
        [
            "--input_dir", str(fsd_dir),
            "--output_dir", str(DEV_SET_DIR / "sound_event"),
            "--pickup_json", str(DEV_SET_DIR / "config" / "FSD50K_config.json"),
            *common,
        ],
    )
    SEMHEAR_ADD_MARKER.write_text("ok", encoding="utf-8")
    print("[done] Semantic Hearing data added")

print("\nStep 4 OK")

## Step 5. 경로 검증

In [ ]:
REQUIRED_PATHS = [
    "data/dev_set/sound_event/train",
    "data/dev_set/noise/train",
    "data/dev_set/interference/train",
    "data/dev_set/room_ir/train",
    "data/dev_set/metadata/valid.json",
    "src/modules/spatial_audio_synthesizer/spatial_audio_synthesizer.py",
]

for rel in REQUIRED_PATHS:
    ok = (PROJECT_ROOT / rel).exists()
    print(f"[{'OK' if ok else 'MISSING'}] {rel}")

missing = [rel for rel in REQUIRED_PATHS if not (PROJECT_ROOT / rel).exists()]
assert not missing, f"Missing paths: {missing}"
print("\nStep 5 OK — all required paths present")

## Step 6. `tests/datamodules` unittest

In [ ]:
def run_datamodule_tests(verbosity=2):
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    suite.addTests(loader.discover(
        start_dir=str(PROJECT_ROOT / "tests" / "datamodules"),
        pattern="test_new_*.py",
        top_level_dir=str(PROJECT_ROOT),
    ))
    stream = StringIO()
    runner = unittest.TextTestRunner(stream=stream, verbosity=verbosity)
    result = runner.run(suite)
    print(stream.getvalue())
    print(f"\nRan {result.testsRun} tests | failures={len(result.failures)} | errors={len(result.errors)}")
    return result

test_result = run_datamodule_tests()
assert not test_result.failures and not test_result.errors, "unittest failed — see output above"
print("\nStep 6 OK — all unittests passed")

## Step 7. Train DataModule 구성 (`new_datamodule` + `new_dataset`)

In [ ]:
from src.datamodules.new_datamodule import DataModule

with open(PROJECT_ROOT / "config" / "label" / "m2dat_1c.yaml") as f:
    cfg = yaml.safe_load(f)

dm_args = cfg["datamodule"]["args"]
dm_args["module"] = "src.datamodules.new_datamodule"
dm_args["main"] = "DataModule"

for split in ("train_dataloader", "val_dataloader"):
    if split not in dm_args:
        continue
    ds_cfg = dm_args[split]["dataset"]
    ds_cfg["module"] = "src.datamodules.new_dataset"
    ds_cfg["main"] = "DatasetS3"

train_cfg = dm_args["train_dataloader"]
train_cfg["batch_size"] = 2
train_cfg["num_workers"] = 0
train_cfg["persistent_workers"] = False
train_cfg["dataset"]["args"]["config"]["dataset_length"] = 8

val_cfg = dm_args.get("val_dataloader")
if val_cfg:
    val_cfg["batch_size"] = 2
    val_cfg["num_workers"] = 0
    val_cfg["persistent_workers"] = False

dm = DataModule(train_dataloader=train_cfg, val_dataloader=val_cfg)
train_loader = dm.train_dataloader()

print("train module:", train_cfg["dataset"]["module"])
print("train mode:", train_cfg["dataset"]["args"]["config"]["mode"])
print("dataset_length:", train_cfg["dataset"]["args"]["config"]["dataset_length"])
print("\nStep 7 OK — DataModule ready")

## Step 8. Train / Val dataloader 스모크 테스트

In [ ]:
batch = next(iter(train_loader))
describe_batch(batch, "train batch")

for key in ("mixture", "labels", "doas", "active"):
    assert key in batch, f"missing {key}"
    assert batch[key].dim() >= 2, f"{key} should be batched"

for step, b in enumerate(train_loader):
    assert b["mixture"].shape[0] <= train_cfg["batch_size"]
    if step >= 2:
        break
print(f"Iterated {step + 1} train batches OK")

if dm.val_dataset is not None:
    val_batch = next(iter(dm.val_dataloader()))
    describe_batch(val_batch, "val batch (metadata mode)")

print("\nStep 8 OK — train dataloader smoke test passed")